In [2]:
import importlib
from copy import deepcopy
from pathlib import Path

import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display

import main
import models.registry
import models.eval.evaluator
import models.architectures.crossmodal_pca_pls

importlib.reload(models.architectures.crossmodal_pca_pls)
importlib.reload(models.eval.evaluator)
importlib.reload(models.registry)
importlib.reload(main)

from main import Sim
from models.architectures.utils import get_model_input
from models.registry import build_model, get_default_config, resolve_source_dependent_config

REPO_ROOT = Path.cwd()
RESULTS_ROOT = Path("results/local_results/crossmodal_pca_pls_closed_form_overview")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

SOURCE = "SC"
TARGET = "FC"
PARCELLATION = "Glasser"
SHUFFLE_SEED = 0
DATA_LOAD_MODE = "precomputed"

CLOSED_FORM_MODELS = {
    "CrossModalPCA": {
        "config": REPO_ROOT / "models/configs/CrossModalPCA.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModalPCA/tune_array_pca_SC_SCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModalPCA/tune_model_single_pca.sh",
        ],
    },
    "CrossModal_PLS_SVD": {
        "config": REPO_ROOT / "models/configs/CrossModal_PLS_SVD.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModal_PLS_SVD/tune_array_pls_svd_SC_SCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModal_PLS_SVD/tune_model_sequential_pls_svd_SC.sh",
        ],
    },
    "CrossModal_PCA_PLS": {
        "config": REPO_ROOT / "models/configs/CrossModal_PCA_PLS.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModal_PCA_PLS/tune_array_pca_pls_SC_SCr2t_SCpSCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModal_PCA_PLS/tune_model_single_pca_pls.sh",
        ],
    },
}

def read_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def show_yaml_default(model_name):
    cfg = read_yaml(CLOSED_FORM_MODELS[model_name]["config"])
    display(Markdown(f"**`{model_name}` config**: `{CLOSED_FORM_MODELS[model_name]['config'].relative_to(REPO_ROOT)}`"))
    display(pd.Series(cfg.get("default", {}), dtype="object"))
    display(Markdown("Search-space keys: " + ", ".join(f"`{k}`" for k in cfg.get("search_space", {}))))


def show_sbatch_refs(model_name):
    refs = CLOSED_FORM_MODELS[model_name]["sbatch"]
    if not refs:
        display(Markdown("No dedicated sbatch launcher is currently present for this model; use `main.py` with its YAML config."))
        return
    lines = [f"- `{p.relative_to(REPO_ROOT)}`" for p in refs]
    display(Markdown("Production launcher references:\n" + "\n".join(lines)))


def model_config_for_notebook(model_name):
    cfg = get_default_config(model_name, path=str(CLOSED_FORM_MODELS[model_name]["config"]))
    cfg.setdefault("data", {})
    cfg["data"].update({
        "source": SOURCE,
        "target": TARGET,
        "parcellation": PARCELLATION,
        "shuffle_seed": SHUFFLE_SEED,
        "data_load_mode": DATA_LOAD_MODE,
    })
    cfg.setdefault("model", {})
    # Closed-form local inspection should stay on CPU. This is an execution-policy override,
    # not a model hyperparameter sweep value.
    cfg["model"]["device"] = "cpu"
    return resolve_source_dependent_config(cfg)


def build_closed_form_model(model_name, base):
    cfg = model_config_for_notebook(model_name)
    model_cfg = deepcopy(cfg["model"])
    model_cfg.pop("name", None)
    return build_model(base, model_name, model_cfg).eval()


def evaluate_closed_form_model(model_name, model):
    out = data_sim._evaluate_model(
        model,
        mode="dev",
        model_name=model_name,
        base=data_sim.base,
        train_loader=data_sim.train_loader,
        val_loader=data_sim.val_loader,
        test_loader=data_sim.test_loader,
    )
    return out


def inspect_one_batch(model, title):
    batch = next(iter(data_sim.val_loader))
    x = get_model_input(batch)
    y = batch["y"]
    with torch.no_grad():
        y_hat = model(x)
    print(title)
    print("device:", next(model.parameters(), torch.empty(0)).device if list(model.parameters()) else "buffer-only")
    print("x shape:", tuple(x.shape) if torch.is_tensor(x) else {k: tuple(v.shape) for k, v in x.items()})
    print("y shape:", tuple(y.shape))
    print("y_hat shape:", tuple(y_hat.shape))
    print("one-batch mse:", float(torch.mean((y_hat.cpu() - y.cpu()) ** 2)))
    return y_hat, y


ModuleNotFoundError: No module named 'main'

In [ ]:
print("hello?")